## Overview

CNN-LSTM Setup for prediction out of 2D timeseries data

finding out of impact of training on "wrong" year of the saison


Colab version of local, loading data from existing dataloaders

----
## Data:

-2D Space - Timeseries

-predicting 1 feature out of 7 variables

-Forecasting 1 timestep (not the following)

-------
Peter Resch, 6.6.

In [1]:
from __future__ import print_function, division   # Ensures Python3 printing & division standard
import pandas as pd
from pandas import Series, DataFrame
from matplotlib import pyplot as plt
import numpy as np
import os

from pathlib import Path

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
import sklearn
from sklearn.model_selection import train_test_split

import xarray as xr
rSeed=42

SavePlots = False

## Loading Data

In [4]:
from google.colab import drive
drive.mount('/content/drive/')

import sys
sys.path.insert(0, '/content/drive/MyDrive/small_grid/') # Add the directory containing the module to the Python path

import my_dataloader_module

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [5]:
!cd drive/MyDrive/small_grid && ls

dataloaders  models  my_dataloader_module.py  __pycache__


In [9]:
seasons={"spring": "MAM", "summer": "JJA", "autumn": "SON", "winter": "DJF"}

data_dir='drive/MyDrive/small_grid'
!cd {data_dir} && ls

dataloader_path = str(data_dir+ "/dataloaders/")

dataloaderlist = [p.name for p in Path(dataloader_path).iterdir() if p.is_file()]
#clean dataloaderlist, let only .pt files
dataloaderlist = [f for f in dataloaderlist if f.endswith('.pt')]
dataloaderlist=[f for f in dataloaderlist if "dataloader" in f]      #select only the ones with "dataloader" in the name

dataloaders  models  my_dataloader_module.py  __pycache__


## Loading Dataloaders

In [10]:
dataloaders = {}
i=0
for dl in dataloaderlist:
    time,aim = dl.split("_dataloader_")
    time,season=time.split("_")
    aim = aim.split(".pt")[0]
    #print(time, season, aim)
    dataloaders[time,season,aim] = torch.load(dataloader_path + time + "_" + season + "_dataloader_"+ aim+".pt", weights_only=False)
    i=i+1
print(i,"Dataloaders loaded successfully.")

20 Dataloaders loaded successfully.


## Building the Neural Network

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")
hidden_size = 16

class CNN_LSTM(nn.Module):
    def __init__(self, in_channels=7, hidden_size=16, lat_size=5, lon_size=5):
        super().__init__()

        self.lat_size = lat_size
        self.lon_size = lon_size
        self.feature_channels = 32

        # Compressing space
        def double_conv(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_ch, out_ch, 3, padding=1),
                nn.ReLU(inplace=True),
            )

        self.conv1 = double_conv(in_channels, 16)
        self.maxpool = nn.MaxPool2d(kernel_size=(2, 2))
        self.conv2 = double_conv(16, self.feature_channels)

        # Force spatial size so lat_size/lon_size are used consistently
        self.spatial_pool = nn.AdaptiveAvgPool2d((lat_size, lon_size))

        # Compressing time with LSTM
        self.seq_input_size = self.feature_channels * lat_size * lon_size
        self.hidden_size = hidden_size if hidden_size is not None else self.seq_input_size

        self.lstm = nn.LSTM(
            input_size=self.seq_input_size,
            hidden_size=self.hidden_size,
            num_layers=1,
            batch_first=True
        )

        # Map LSTM hidden state back to spatial tensor size
        self.proj = nn.Linear(self.hidden_size, self.seq_input_size)

        # Expanding space
        self.deconv1 = nn.ConvTranspose2d(self.feature_channels, 16, kernel_size=3, padding=1)
        self.deconv2 = nn.ConvTranspose2d(16, 1, kernel_size=3, padding=1)

    def forward(self, x):
        x_seq = []
        for t in range(x.shape[1]):
            cnn_in = x[:, t, :, :]
            cnn_in = self.conv1(cnn_in)
            cnn_in = self.maxpool(cnn_in)
            cnn_in = self.conv2(cnn_in)
            cnn_in = self.maxpool(cnn_in)
            cnn_in = self.spatial_pool(cnn_in)
            cnn_in = cnn_in.view(cnn_in.size(0), -1)  # (batch, seq_input_size)
            x_seq.append(cnn_in)

        x_seq = torch.stack(x_seq, dim=1)  # (batch, time, seq_input_size)

        # LSTM over time
        x, _ = self.lstm(x_seq)
        x = x[:, -1, :]                    # last timestep
        x = self.proj(x)                   # (batch, seq_input_size)
        x = x.view(x.size(0), self.feature_channels, self.lat_size, self.lon_size)

        x = self.deconv1(x)
        x = self.deconv2(x)
        return x

# Keep variable name for downstream cells compatibility
cnn_lstm_model = CNN_LSTM(
    hidden_size=hidden_size,
    lat_size=5,
    lon_size=5
).to(device)

print(cnn_lstm_model)


Using cuda device


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:1364: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("GRU", *args, **kwargs)


CNN_GRU(
  (conv1): Sequential(
    (0): Conv2d(7, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
  )
  (maxpool): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
  (conv2): Sequential(
    (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
  )
  (gru): GRU(32, 800, batch_first=True, dropout=0.2)
  (deconv1): ConvTranspose2d(32, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (deconv2): ConvTranspose2d(16, 1, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
)


In [12]:
def train(dataloader, model, loss_fn, optimizer,device):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        #print(X.shape, y.shape)
        X, y = X.to(device), y.to(device)
        #print(X.shape, y.shape)
        pred = model(X)#.squeeze()
        #print(pred)#.shape)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            #print(f"Train Loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
    return loss.item()



def test(dataloader, model, loss_fn,device):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss = 0.0
    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            predictions = model(X)
            test_loss += loss_fn(predictions, y).item()

            all_predictions.append(predictions.detach().cpu().numpy())
            all_targets.append(y.detach().cpu().numpy())

    test_loss /= num_batches
    y_pred = np.concatenate(all_predictions)#[:,:,0]
    y_true = np.concatenate(all_targets)#[:,:,0]
    #print(f"y_true shape: {y_true.shape}, y_pred shape: {y_pred.shape}")

    # Flatten everything to 2D: (samples, features)
    y_true_flat = y_true.reshape(y_true.shape[0], -1)
    y_pred_flat = y_pred.reshape(y_pred.shape[0], -1)

    #mae = sklearn.metrics.mean_absolute_error(y_true, y_pred)
    #rmse = np.sqrt(sklearn.metrics.mean_squared_error(y_true, y_pred))
    r2 = sklearn.metrics.r2_score(y_true_flat, y_pred_flat)
    print(f"Test Error:\n R2: {r2:>8f}, Avg loss: {test_loss:>8f} \n")
    return test_loss

## Train the model

In [ ]:
time = "recent"

epochs =50


for season in seasons.keys():
    print(":"*50)
    print(f"Season: {season}")
    model_name = f"cnn_lstm_{time}_{season}"

    loss_fcn = nn.MSELoss()
    optimizer_cnn_lstm = AdamW(cnn_lstm_model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_cnn_lstm, mode="min", factor=0.5, patience=2
    )
    train_losses = []
    val_losses = []
    test_losses = []

    for t in range(epochs):
        print(f"Epoch {t+1}\n-------------------------------")
        train_loss = train(dataloaders[time, season, "train"], cnn_lstm_model, loss_fcn, optimizer_cnn_lstm, device)
        val_loss = test(dataloaders[time, season, "val"], cnn_lstm_model, loss_fcn, device)
        test_loss = test(dataloaders["now", season, "test"], cnn_lstm_model, loss_fcn, device)

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        test_losses.append(test_loss)

        scheduler.step(val_loss)

    train_losses = np.array(train_losses)
    val_losses = np.array(val_losses)
    test_losses = np.array(test_losses)

    print("Training done!")

    os.makedirs("models", exist_ok=True)
    save_path = f"{data_dir}/models/{model_name}.pth"

    torch.save({
        "model_state_dict": cnn_lstm_model.state_dict(),
        "optimizer_state_dict": optimizer_cnn_lstm.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "lag_selection": [0, 24, 48, 60, 66, 69, 71],
        "target_var": 4,
        "epoch": epochs
    }, save_path)

    print(f"Saved model to {save_path}")

    df = pd.DataFrame({
        "Train Loss": train_losses,
        "Validate Loss": val_losses,
        "Test Loss": test_losses
    })
    df.to_csv(f"{data_dir}/models/losses_{model_name}.csv", index=False)
    print(f"Saved losses to losses_{model_name}.csv")

    print(f"Finished traininig {model_name} for {epochs} epochs.")
    print("-"*30)



::::::::::::::::::::::::::::::::::::::::::::::::::
Season: spring
Epoch 1
-------------------------------
Test Error:
 R2: 0.437540, Avg loss: 0.619455 

Test Error:
 R2: 0.437722, Avg loss: 0.581322 

Epoch 2
-------------------------------
Test Error:
 R2: 0.493044, Avg loss: 0.558314 

Test Error:
 R2: 0.496852, Avg loss: 0.522543 

Epoch 3
-------------------------------
Test Error:
 R2: 0.519329, Avg loss: 0.529027 

Test Error:
 R2: 0.516649, Avg loss: 0.499738 

Epoch 4
-------------------------------
Test Error:
 R2: 0.540137, Avg loss: 0.507046 

Test Error:
 R2: 0.535975, Avg loss: 0.480571 

Epoch 5
-------------------------------
Test Error:
 R2: 0.564908, Avg loss: 0.481457 

Test Error:
 R2: 0.566870, Avg loss: 0.450633 

Epoch 6
-------------------------------
Test Error:
 R2: 0.581405, Avg loss: 0.465745 

Test Error:
 R2: 0.578061, Avg loss: 0.442538 

Epoch 7
-------------------------------
Test Error:
 R2: 0.581445, Avg loss: 0.464934 

Test Error:
 R2: 0.589045, Avg